In [1]:
# %pip install pyspark

In [2]:
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder.appName("AmExDefaultPrediction")
    .master("local[*]")  # Use all available cores on your machine/Colab runtime
    .getOrCreate()
)

print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/15 17:07:02 WARN Utils: Your hostname, Vineel-Kotapatis-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.103 instead (on interface en0)
25/11/15 17:07:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/15 17:07:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.0.1'

In [4]:
df_features = spark.read.csv(
    "data/train_data.csv", 
    header=True, 
    inferSchema=True
)

In [5]:
len(df_features.columns)

190

In [6]:
df_features.count()

5531451

In [7]:
df_features.show(5)

25/11/15 17:08:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+----------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+----+----+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+----+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+----+------------------+------------------

In [8]:
df_labels = spark.read.csv(
    "data/train_labels.csv", 
    header=True, 
    inferSchema=True
)

In [9]:
df_labels.count()

458913

In [10]:
df_labels.show(5)

+--------------------+------+
|         customer_ID|target|
+--------------------+------+
|0000099d6bd597052...|     0|
|00000fd6641609c6e...|     0|
|00001b22f846c82c5...|     0|
|000041bdba6ecadd8...|     0|
|00007889e4fcd2614...|     0|
+--------------------+------+
only showing top 5 rows


In [11]:
df = df_features.join(
    df_labels, 
    on="customer_ID", 
    how="inner"
)
df.show(5)

+--------------------+----------+------------------+------------------+--------------------+------------------+------------------+----+--------------------+------------------+------------------+----+-------------------+------------------+------------------+--------------------+------------------+------------------+------------------+------------------+----+------------------+------------------+------------------+----+------------------+------------------+------------------+------------------+------------------+------------------+----+--------------------+-------------------+------------------+------------------+--------------------+----+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+----+----+------------------+------------------+----+------------------+----------

In [12]:
df.printSchema()

root
 |-- customer_ID: string (nullable = true)
 |-- S_2: date (nullable = true)
 |-- P_2: double (nullable = true)
 |-- D_39: double (nullable = true)
 |-- B_1: double (nullable = true)
 |-- B_2: double (nullable = true)
 |-- R_1: double (nullable = true)
 |-- S_3: double (nullable = true)
 |-- D_41: double (nullable = true)
 |-- B_3: double (nullable = true)
 |-- D_42: double (nullable = true)
 |-- D_43: double (nullable = true)
 |-- D_44: double (nullable = true)
 |-- B_4: double (nullable = true)
 |-- D_45: double (nullable = true)
 |-- B_5: double (nullable = true)
 |-- R_2: double (nullable = true)
 |-- D_46: double (nullable = true)
 |-- D_47: double (nullable = true)
 |-- D_48: double (nullable = true)
 |-- D_49: double (nullable = true)
 |-- B_6: double (nullable = true)
 |-- B_7: double (nullable = true)
 |-- B_8: double (nullable = true)
 |-- D_50: double (nullable = true)
 |-- D_51: double (nullable = true)
 |-- B_9: double (nullable = true)
 |-- R_3: double (nullable = tru

In [13]:
df.count()

5531451

In [14]:
len(df.columns)

191

In [15]:
df.describe().show()

25/11/15 17:14:34 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


+-------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-

In [16]:
print("Target variable distribution:")
total_rows = df.count()
(
    df.groupBy("target")
    .count()
    .show()
)

Target variable distribution:


+------+-------+
|target|  count|
+------+-------+
|     1|1377869|
|     0|4153582|
+------+-------+



In [17]:
columns = df.columns
cat_cols = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68', 'D_63']
date_cols = ['S_2']
id_cols = ['customer_ID']
target_col = ['target']
num_cols = [col for col in columns if col not in cat_cols + id_cols + date_cols + target_col]

In [18]:
## Remove unnecessary columns
cols_to_drop = ["Customer_ID", "S_2"]
df = df.drop(*cols_to_drop)

In [19]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from typing import List

# 1. Get the DataFrame after imputation
# (Assuming this is df_imputed from our previous step)
df_to_index = df 

# 2. Automatically get all string columns from the schema
cat_cols = [
    col_name for col_name, data_type in df_to_index.dtypes 
    if data_type == 'string'
]

# (Optional) Print the columns we found
print(f"Found {len(cat_cols)} string columns to index: {cat_cols}")

# 3. Create a list of StringIndexer stages (with type hint)
indexer_stages = [
    StringIndexer(
        inputCol=col, 
        outputCol=col+"_index",  # Overwrite the original column
        handleInvalid="keep"
    ) 
    for col in cat_cols
]

# 4. Create the Pipeline
pipeline = Pipeline(stages=indexer_stages) # type: ignore

# 5. Fit and transform the data
print("Fitting StringIndexers and replacing columns...")
pipeline_model = pipeline.fit(df_to_index)
df_indexed = pipeline_model.transform(df_to_index)

# 6. Check the schema to confirm
print("\nSchema after in-place indexing:")
df_indexed.printSchema()

df_indexed = df_indexed.drop(*cat_cols)

# (Optional) Show a few of the newly indexed columns
if cat_cols:
    print(f"\nShowing indexed columns: {cat_cols[:3]}")
    df_indexed.select([x+"_index" for x in cat_cols[:3]]).show(5)

Found 2 string columns to index: ['D_63', 'D_64']
Fitting StringIndexers and replacing columns...



Schema after in-place indexing:
root
 |-- P_2: double (nullable = true)
 |-- D_39: double (nullable = true)
 |-- B_1: double (nullable = true)
 |-- B_2: double (nullable = true)
 |-- R_1: double (nullable = true)
 |-- S_3: double (nullable = true)
 |-- D_41: double (nullable = true)
 |-- B_3: double (nullable = true)
 |-- D_42: double (nullable = true)
 |-- D_43: double (nullable = true)
 |-- D_44: double (nullable = true)
 |-- B_4: double (nullable = true)
 |-- D_45: double (nullable = true)
 |-- B_5: double (nullable = true)
 |-- R_2: double (nullable = true)
 |-- D_46: double (nullable = true)
 |-- D_47: double (nullable = true)
 |-- D_48: double (nullable = true)
 |-- D_49: double (nullable = true)
 |-- B_6: double (nullable = true)
 |-- B_7: double (nullable = true)
 |-- B_8: double (nullable = true)
 |-- D_50: double (nullable = true)
 |-- D_51: double (nullable = true)
 |-- B_9: double (nullable = true)
 |-- R_3: double (nullable = true)
 |-- D_52: double (nullable = true)
 |--

+----------+----------+
|D_63_index|D_64_index|
+----------+----------+
|       0.0|       2.0|
|       0.0|       2.0|
|       0.0|       2.0|
|       0.0|       2.0|
|       0.0|       2.0|
+----------+----------+
only showing top 5 rows


In [25]:
df_indexed = df_indexed.withColumnsRenamed(
    {col+"_index": col for col in cat_cols})

In [26]:
from pyspark.sql.functions import col, isnan, when, count

# 1. Get a list of (column_name, data_type) tuples
col_types = df.dtypes

missing_counts = []

total_rows = df.count()

# 2. Iterate over columns and apply the correct logic
for c, dtype in col_types:
    if dtype in ('double', 'float'):
        # Numeric columns: check for both NaN and Null
        check = count(when(isnan(c) | col(c).isNull(), c)).alias(c)
    elif dtype == 'string':
        # String columns: only check for Null
        check = count(when(col(c).isNull(), c)).alias(c)
    else:
        # Other types (int, boolean, timestamp, etc.): only check for Null
        check = count(when(col(c).isNull(), c)).alias(c)
        
    missing_counts.append(check)

# 3. Run the aggregation (this part is the same)
missing_df = df_indexed.select(missing_counts).first()

# 4. Print the results cleanly
print("Missing values per column:")
missing_dict = {}
if missing_df:
    for col_name, count_val in missing_df.asDict().items():
        if count_val > 0:
            print(f"{col_name}: {count_val/total_rows:.2%} missing")
            missing_dict[col_name] = [count_val, count_val/total_rows]

print("\nDone checking all columns.")

Missing values per column:
P_2: 0.83% missing
B_2: 0.04% missing
S_3: 18.45% missing
D_41: 0.04% missing
B_3: 0.04% missing
D_42: 85.69% missing
D_43: 29.98% missing
D_44: 4.96% missing
D_45: 0.04% missing
D_46: 21.91% missing
D_48: 12.99% missing
D_49: 90.14% missing
B_6: 0.00% missing
B_8: 0.40% missing
D_50: 56.81% missing
D_52: 0.53% missing
P_3: 5.45% missing
D_53: 73.84% missing
D_54: 0.04% missing
S_7: 18.45% missing
D_55: 3.34% missing
D_56: 54.07% missing
B_13: 0.90% missing
S_9: 53.04% missing
D_59: 1.93% missing
D_61: 10.81% missing
B_15: 0.13% missing
D_62: 13.71% missing
B_16: 0.04% missing
B_17: 56.72% missing
B_19: 0.04% missing
D_66: 88.73% missing
B_20: 0.04% missing
D_68: 3.91% missing
D_69: 3.52% missing
B_22: 0.04% missing
D_70: 1.72% missing
D_72: 0.43% missing
D_73: 98.99% missing
D_74: 0.39% missing
D_76: 88.75% missing
R_7: 0.00% missing
D_77: 45.45% missing
B_25: 0.13% missing
B_26: 0.04% missing
D_78: 4.96% missing
D_79: 1.37% missing
R_9: 94.35% missing
D_80:

In [27]:
## Remove columns with more than 90% missing values for numerical columns
cols_to_drop = [col for col, miss_pct in missing_dict.items() if miss_pct[1] > 0.9 and col in num_cols]
print("Columns to drop due to >90% missing values in numerical columns:")
print(cols_to_drop)
df_cleaned = df_indexed.drop(*cols_to_drop)

Columns to drop due to >90% missing values in numerical columns:
['D_49', 'D_73', 'R_9', 'B_29', 'D_87', 'D_88', 'D_106', 'D_108', 'D_110', 'D_111', 'B_39', 'B_42', 'D_132', 'D_134', 'D_135', 'D_136', 'D_137', 'D_138']


In [28]:
## trying removing columns with more than 50% missing values for numerical columns
cols_to_drop = [col for col, miss_pct in missing_dict.items() if miss_pct[1] > 0.5 and col in num_cols]
print("Columns to drop due to >50% missing values in numerical columns:")
print(cols_to_drop)
df_cleaned = df_cleaned.drop(*cols_to_drop)

Columns to drop due to >50% missing values in numerical columns:
['D_42', 'D_49', 'D_50', 'D_53', 'D_56', 'S_9', 'B_17', 'D_73', 'D_76', 'R_9', 'D_82', 'B_29', 'D_87', 'D_88', 'D_105', 'D_106', 'R_26', 'D_108', 'D_110', 'D_111', 'B_39', 'B_42', 'D_132', 'D_134', 'D_135', 'D_136', 'D_137', 'D_138', 'D_142']


In [29]:
## trying removing columns with more than 50% missing values for categorical columns
cols_to_drop = [col for col, miss_pct in missing_dict.items() if miss_pct[1] > 0.5 and col in cat_cols]
print("Columns to drop due to >50% missing values in categorical columns:")
print(cols_to_drop)
df_cleaned = df_cleaned.drop(*cols_to_drop)

Columns to drop due to >50% missing values in categorical columns:
[]


In [30]:
len(df_cleaned.columns)

160

In [31]:
## checking missing values in categorical columns after dropping one column D_66

for x, y in missing_dict.items():
    if x in cat_cols:
        print(x, y)

In [32]:
impute_cols = [col for col in df_cleaned.columns if col in missing_dict]
len(impute_cols)

92

In [33]:
from pyspark.ml.feature import Imputer

for i in range(0, len(impute_cols), 21):

    impute_num_cols = impute_cols[i:i+21]

    # 2. Create the Imputer
    imputer = Imputer(
        strategy='median',  # You can also use 'mean'
        inputCols=impute_num_cols,
        outputCols=impute_num_cols  # This will overwrite the original columns
    )

    # 3. "Fit" the imputer: This action calculates the median for each column
    imputer_model = imputer.fit(df_cleaned)
    df_imputed = imputer_model.transform(df_cleaned)

[6394.305s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 126.0 (TID 2508): Retried waiting for GCLocker too often allocating 16825 words
[6394.305s][warning][gc,alloc] Executor task launch worker for task 9.0 in stage 126.0 (TID 2516): Retried waiting for GCLocker too often allocating 8295 words
[6395.158s][warning][gc,alloc] Executor task launch worker for task 4.0 in stage 126.0 (TID 2511): Retried waiting for GCLocker too often allocating 16568 words
[6395.161s][warning][gc,alloc] Executor task launch worker for task 11.0 in stage 126.0 (TID 2518): Retried waiting for GCLocker too often allocating 24529 words
[6395.161s][warning][gc,alloc] Executor task launch worker for task 0.0 in stage 126.0 (TID 2507): Retried waiting for GCLocker too often allocating 11771 words


In [34]:
from pyspark.ml.feature import VectorAssembler

# 1. Get our fully preprocessed DataFrame
final_df = df_imputed

# 2. Automatically get all feature column names
# We'll take every column *except* 'target' and 'customer_ID'
# (and any other original string/ID columns we want to exclude)
feature_cols = [
    col for col, dtype in final_df.dtypes 
    if col not in ('customer_ID', 'target') and dtype != 'string'
]

# (Optional) Check the first 10 feature columns
print(f"Assembling {len(feature_cols)} features.")
print(f"First 10 features: {feature_cols[:10]}")

# 3. Create the VectorAssembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"  # Will keep rows with nulls (though we imputed them)
)

# 4. Transform the DataFrame
# This is a fast operation as it's just combining columns
print("\nAssembling features into a single vector...")
model_ready_df = assembler.transform(final_df)

# 5. Show the final result
print("Final DataFrame ready for ML:")
model_ready_df.select("target", "features").show(5, truncate=False)

Assembling 159 features.
First 10 features: ['P_2', 'D_39', 'B_1', 'B_2', 'R_1', 'S_3', 'D_41', 'B_3', 'D_43', 'D_44']

Assembling features into a single vector...
Final DataFrame ready for ML:


+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [36]:
# Our final, ML-ready DataFrame
df = model_ready_df

# Split the data into 80% training and 20% testing
(training_data, test_data) = df.randomSplit([0.8, 0.2], seed=42)

# Optional: Cache the data in memory for faster training
training_data.cache()

print(f"Total rows: {df.count()}")
print(f"Training rows: {training_data.count()}")
# print(f"Test rows: {test_data.count()}")

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# 1. Create the classifier
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="target",
    seed=42
)

# 2. Train the model
# This is the action that will take the most time
print("Training Random Forest model...")
rf_model = rf.fit(training_data)

print("Random Forest training complete.")

In [ ]:


from pyspark.ml.evaluation import BinaryClassificationEvaluator

training_data.unpersist()
test_data.cache()

rf_predictions = rf_model.transform(test_data)

# Create an evaluator that looks at the 'target' column
# By default, it uses the 'rawPrediction' column for AUC
auc_evaluator = BinaryClassificationEvaluator(labelCol="target")

# Calculate AUC for both models
rf_auc = auc_evaluator.evaluate(rf_predictions)

print(f"Random Forest AUC: {rf_auc:.4f}")

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create evaluators for each metric
# We must specify the 'prediction' and 'target' columns
precision_eval = MulticlassClassificationEvaluator(
    labelCol="target", predictionCol="prediction", metricName="weightedPrecision"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="target", predictionCol="prediction", metricName="weightedRecall"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="target", predictionCol="prediction", metricName="f1"
)

# Calculate metrics for Random Forest
rf_precision = precision_eval.evaluate(rf_predictions)
rf_recall = recall_eval.evaluate(rf_predictions)
rf_f1 = f1_eval.evaluate(rf_predictions)

print("\n--- Random Forest ---")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall: {rf_recall:.4f}")
print(f"F1-Score: {rf_f1:.4f}")

In [ ]:
# %pip install xgboost

  Using cached xgboost-3.1.1-py3-none-macosx_10_15_x86_64.whl.metadata (2.1 kB)
Using cached xgboost-3.1.1-py3-none-macosx_10_15_x86_64.whl (2.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# # This requires 'pip install xgboost'
# from xgboost.spark import SparkXGBClassifier

# # 1. Create the XGBoost classifier
# # It's very important to set featuresCol and labelCol
# xgb = SparkXGBClassifier(
#     features_col="features",
#     label_col="target",
#     seed=42
# )

# # 2. Train the model
# print("\nTraining XGBoost model...")
# xgb_model = xgb.fit(training_data)

# print("XGBoost training complete.")

In [ ]:
# from pyspark.ml.evaluation import BinaryClassificationEvaluator


# xgb_predictions = xgb_model.transform(test_data)

# # Create an evaluator that looks at the 'target' column
# # By default, it uses the 'rawPrediction' column for AUC
# auc_evaluator = BinaryClassificationEvaluator(labelCol="target")

# # Calculate AUC for both models
# xgb_auc = auc_evaluator.evaluate(xgb_predictions)

# print(f"XGBoost AUC: {xgb_auc:.4f}")

In [ ]:
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# # Create evaluators for each metric
# # We must specify the 'prediction' and 'target' columns
# precision_eval = MulticlassClassificationEvaluator(
#     labelCol="target", predictionCol="prediction", metricName="weightedPrecision"
# )
# recall_eval = MulticlassClassificationEvaluator(
#     labelCol="target", predictionCol="prediction", metricName="weightedRecall"
# )
# f1_eval = MulticlassClassificationEvaluator(
#     labelCol="target", predictionCol="prediction", metricName="f1"
# )

# # Calculate metrics for XGBoost
# xgb_precision = precision_eval.evaluate(xgb_predictions)
# xgb_recall = recall_eval.evaluate(xgb_predictions)
# xgb_f1 = f1_eval.evaluate(xgb_predictions)

# print("\n--- XGBoost ---")
# print(f"Precision: {xgb_precision:.4f}")
# print(f"Recall: {xgb_recall:.4f}")
# print(f"F1-Score: {xgb_f1:.4f}")